# 06d — Matched shared latent-propagator and spatial-GNN rollout

Compare the shared three-source latent propagator from 06b with one shared spatial GNN. Both models use the same shuffled trajectories, start from frame 0, receive no source label or temperature, and are scored on the same held-out networks. Real Reid uses the validation-selected cumulative side-strain estimator from 06b for both models; the other sources retain the directional-side endpoint estimator.

In [1]:
%matplotlib inline
import os, sys
from pathlib import Path
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT/'src'/'lss').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT/'src') not in sys.path: sys.path.insert(0, str(PROJECT_ROOT/'src'))
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from IPython.display import display
from lss.latent.capacity import load_experiment_bundle
from lss.matched_gnn import run_matched_shared_spatial_gnn
from lss.plotting import PAPER_COLORS, apply_editorial_style, dataset_color, style_axes
from lss.utils import resolve_device
apply_editorial_style()
PAPER_DPI = 180
PAPER_FIGSIZE = (5.4, 4.2)
PAPER_WIDE_FIGSIZE = (10.8, 4.2)
plt.rcParams.update({'figure.dpi': PAPER_DPI, 'savefig.dpi': 400})
DEVICE = resolve_device('auto')
DEVICE

/home/alexz/Documents/course_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'cuda'

## Matched configuration

In [2]:
FORCE_TRAIN_GNN = False
GNN_EPOCHS = 40
EVAL_NETWORKS_PER_SOURCE = 30
ROLLOUT_STEPS = [5, 10, 25, 50, 75, 100, 125, 150, 175, 199]
LATENT_CACHE = PROJECT_ROOT/'notebooks'/'results'/'06b_mixed_dataset_shared_latent_rollout_boxnorm'/'model_compact_edges_v5_matched_budget_history_z1_z5.pt'
OUTPUT = PROJECT_ROOT/'notebooks'/'results'/'06d_matched_shared_gnn_rollout_comparison_boxnorm'
OUTPUT.mkdir(parents=True, exist_ok=True)
assert LATENT_CACHE.exists(), 'Run notebook 06b first: '+str(LATENT_CACHE)
SOURCE_STYLES = {
    'reid': {'label': 'Reid', 'marker': 's'},
    'depablo_low_temp': {'label': 'de Pablo low-T', 'marker': 'o'},
    'depablo_mixed_temp': {'label': 'de Pablo mixed-T', 'marker': '^'},
}
# Reserve dashed lines for the direct-autoencoder ceiling in 06b. Here,
# model identity is carried by marker shape/fill and remains legible in grayscale.
MODEL_STYLES = {
    'latent propagator': {'linestyle': '-', 'marker': 'D', 'markerfacecolor': 'none', 'markeredgewidth': 1.25, 'lw': 2.0, 'ms': 5.5},
    'spatial GNN': {'linestyle': '-', 'marker': 'X', 'markerfacecolor': 'color', 'markeredgecolor': 'white', 'markeredgewidth': .55, 'lw': 1.45, 'ms': 6.0},
}
print({'GNN epochs': GNN_EPOCHS, 'held-out networks per source': EVAL_NETWORKS_PER_SOURCE, 'rollout steps': ROLLOUT_STEPS})

AssertionError: Run notebook 06b first: /home/alexz/Documents/course_project/notebooks/results/06b_mixed_dataset_shared_latent_rollout_boxnorm/model.pt

## Load the cached shared latent propagator

In [ ]:
latent_result = load_experiment_bundle(LATENT_CACHE, experiment_config={}, device=DEVICE)
latent_params = latent_result['params']
assert latent_params['propagator_objective'] == 'fixed_history_one_step'
assert tuple(latent_params['fixed_observed_frames']) == (1, 5)
assert int(latent_params['dyn_max_train_transitions_per_sim']) == 100
display(latent_result['split_info'])
print({
    'split seed': latent_params['split_seed'],
    'training networks per source': [item['train_count'] for item in latent_params['dataset_mixture']],
    'training transitions per network': latent_params['dyn_max_train_transitions_per_sim'],
    'latent observations': latent_params['fixed_observed_frames'], 'GNN history': 0,
})

## Shared spatial-GNN comparison

One spatial GNN is trained across all three sources, exactly like the shared latent model. The helper verifies the independently loaded GNN train and validation splits against the cached latent trajectories before training or evaluation.

In [ ]:
comparison = run_matched_shared_spatial_gnn(
    latent_result=latent_result, rollout_steps=ROLLOUT_STEPS,
    eval_count_per_source=EVAL_NETWORKS_PER_SOURCE, device=DEVICE,
    output_dir=OUTPUT, epochs=GNN_EPOCHS,
    force_train=FORCE_TRAIN_GNN, history=0, reid_trajectory_pratio=True,
)
comparison.to_csv(OUTPUT/'matched_shared_latent_vs_spatial_gnn.csv', index=False)
display(comparison.round(5))

## Held-out rollout performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=PAPER_WIDE_FIGSIZE, constrained_layout=True)
for source, source_style in SOURCE_STYLES.items():
    source_rows = comparison[comparison.source.eq(source)]
    for model_name, model_style in MODEL_STYLES.items():
        group = source_rows[source_rows.model.eq(model_name)].sort_values('rollout_steps')
        color = dataset_color(source)
        common = dict(
            color=color, linestyle=model_style['linestyle'], marker=model_style['marker'],
            markerfacecolor=color if model_style['markerfacecolor'] == 'color' else model_style['markerfacecolor'],
            markeredgecolor=model_style.get('markeredgecolor', color),
            markeredgewidth=model_style['markeredgewidth'], lw=model_style['lw'], ms=model_style['ms'],
        )
        axes[0].plot(group.rollout_steps, group.p_ratio_r2, **common)
        axes[1].plot(group.rollout_steps, group.position_mse, **common)
axes[0].axhline(0, color=PAPER_COLORS['ink'], lw=.7, alpha=.5)
style_axes(axes[0], xlabel='step', ylabel='p-ratio R²', legend=False)
style_axes(axes[1], xlabel='step', ylabel='MSE', legend=False)
axes[0].set(xlim=(0, max(ROLLOUT_STEPS)), ylim=(-1, 1))
axes[1].set_xlim(0, max(ROLLOUT_STEPS)); axes[1].set_yscale('log')
for ax in axes: ax.set_xticks([0, 50, 100, 150, 199])
dataset_handles = [Line2D([0], [0], color=dataset_color(source), lw=2, label=style['label']) for source, style in SOURCE_STYLES.items()]
MODEL_LABELS = {'latent propagator': 'LSS', 'spatial GNN': 'GNN'}
model_handles = [Line2D(
    [0, 1], [0, 0], color=PAPER_COLORS['ink'], linestyle=style['linestyle'],
    marker=style['marker'], markerfacecolor=PAPER_COLORS['ink'] if style['markerfacecolor'] == 'color' else style['markerfacecolor'],
    markeredgecolor='white' if style.get('markeredgecolor') == 'white' else PAPER_COLORS['ink'],
    markeredgewidth=style['markeredgewidth'], markersize=style['ms'], lw=style['lw'], label=MODEL_LABELS[name],
) for name, style in MODEL_STYLES.items()]
dataset_legend = axes[1].legend(handles=dataset_handles, loc='lower right', bbox_to_anchor=(.99, .18), frameon=False, borderaxespad=.2, labelspacing=.35)
axes[1].add_artist(dataset_legend)
axes[1].legend(handles=model_handles, loc='lower right', bbox_to_anchor=(.99, .02), frameon=False, borderaxespad=.2, labelspacing=.35, handlelength=3.0, handletextpad=.65)
plt.show()

## Noisy-LJ feasibility with the learned 2D latent

Noisy LJ is a harder stochastic case. The direct 2D-AE ceiling remains useful near frame 100 but deteriorates by frame 199, so this diagnostic makes the supported claim only at frame 100. It uses the cached held-out history-spacing sweep from notebook 08: the propagator is trained on noisy-LJ training trajectories and receives learned $z(1)$ and $z(k)$, without a frame-progress input. This is not de-Pablo-to-noisy dynamics transfer.

In [ ]:
NOISY_RESULTS = PROJECT_ROOT/'notebooks/results/08_history_aware_latent_rollout/lj_noisy/direct4_displacement_ae_rolling_latent_history'
history_candidates = sorted(NOISY_RESULTS.glob('latent_dimension_history_sweep/**/frame100_history_sweep_test.csv'))
curve_candidates = sorted(NOISY_RESULTS.glob('latent_dimension_comparison/**/ae_ceiling_and_rollout_r2_by_dimension_and_step.csv'))
if not history_candidates or not curve_candidates:
    raise FileNotFoundError('Run the latent-dimension and history-spacing cells in notebook 08 first.')
history_table = pd.read_csv(history_candidates[-1])
history_2d = history_table[history_table.latent_dim.eq(2) & history_table.split.eq('test') & history_table.k.le(5)].sort_values('k')
curve_table = pd.read_csv(curve_candidates[-1])
ceiling_row = curve_table[curve_table.latent_dim.eq(2) & curve_table.step.eq(100) & curve_table.measurement.eq('AE ceiling')]
if history_2d.empty or ceiling_row.empty:
    raise ValueError('The cached noisy-LJ tables do not contain the required held-out 2D results.')
ceiling_r2 = float(ceiling_row.p_ratio_r2.iloc[-1])
best = history_2d.loc[history_2d.p_ratio_r2.idxmax()]
noisy_lj_summary = pd.DataFrame([
    {'measurement':'direct Autoencoder','observed history':'—','frame':100,'p_ratio_r2':ceiling_r2},
    {'measurement':'best 2D history rollout','observed history':f"z(1), z({int(best.k)})",'frame':100,'p_ratio_r2':float(best.p_ratio_r2)},
])
display(noisy_lj_summary.round(4))
fig, ax = plt.subplots(figsize=PAPER_FIGSIZE, constrained_layout=True)
color = dataset_color('lj_noisy')
ax.plot(history_2d.k, history_2d.p_ratio_r2, color=color, marker='o', ms=4.2, lw=1.9, label='2D history rollout')
ax.axhline(ceiling_r2, color=PAPER_COLORS['slate'], ls='--', lw=1.8, label='direct Autoencoder')
ax.scatter([best.k], [best.p_ratio_r2], color=color, marker='s', s=42, zorder=4)
style_axes(ax, xlabel='second observed frame k', ylabel=r'frame-100 p-ratio $R^2$', legend=False)
ax.legend(frameon=False, loc='lower right')
plt.show()